# Calculating the Fisher forecasts

This notebook reproduces every Fisher forecast in Cheslog et al. (2026). It
computes the numerical derivatives of the CMB and BAO theory, builds the
twenty Fisher matrices from them, and reports the forecasted parameter
errors that go into the figures and tables.

It follows the `example_calculate_fisher_matrices.ipynb` notebook in
[hdfisher](https://github.com/CMB-HD/hdfisher), which is the place to start
if you have not used the `Fisher` class before. What we add here are the
nine configurations this paper needs.

Most of the forecasts in the paper are made with CLASS. That includes
Table IV, Figures 2 and 4, and the extended models. CAMB is used where
CLASS cannot be, or where the two codes disagree. Those cases are the
binned $\mathcal{P}(k)$ forecasts, which go through hdPk and CAMB, the
$w_0 w_a$CDM forecasts, where we found CAMB and CLASS differ by up to about
45%, and the baryonic feedback forecast of Table V. A CAMB nine-parameter
forecast is also kept for the comparison with CLASS in Appendix B and for
the accuracy study of Appendix A.

The derivatives are the expensive part. They can be run from this notebook,
and the section on the nine configurations shows how, but one configuration
takes hours at our accuracy settings and there are nine. In practice
`run_hdInitPk_forecasts.py` under MPI is the way to run them. Everything
after that takes seconds and is done here.

## Setup

The step-size and fiducial-parameter files ship with the package, so these
paths work wherever `hdinitpk` is installed. The derivatives are written to
`hdinitpk/data/user_generated_data`, since they are far too large to ship
with the package.

In [ ]:
import os

import numpy as np
import pandas as pd

from hdfisher import fisher
from hdfisher import fisher as hdfisher_fisher
from hd_mock_data import hd_data
from hdinitpk import hdinitPkfisher as hdinitpk_fisher
import hdinitpk

steps_dir = hdinitpk.data_path('fisher_steps')
params_dir = hdinitpk.data_path('fisher_fid_params')
saved_matrix_dir = hdinitpk.data_path('fisher_matrices')

# Output goes under `hdinitpk/data/user_generated_data`, the same place
# `run_hdInitPk_forecasts.py` writes, so the two are interchangeable. Set
# HDINITPK_USER_DATA to move it elsewhere.
deriv_dir = hdinitpk.user_data_path('fisher_derivs')
matrix_out_dir = hdinitpk.user_data_path('fisher_matrices')


def steps(name):
    """A step-size file from the package."""
    return os.path.join(steps_dir, name)


def params(name):
    """A fiducial-parameter file from the package.

    The CLASS files name the sBBN table, and CLASS opens that file itself,
    so the `/path/to/hdInitPk` placeholder in the shipped copy has to be
    replaced with a real path. The rewritten copy goes to
    `user_generated_data/resolved_params`, since the package directory
    may be read-only and should not be edited in place anyway.
    """
    src = os.path.join(params_dir, name)
    with open(src, encoding='utf-8') as f:
        text = f.read()
    placeholder = '/path/to/hdInitPk/hdinitpk/data'
    if placeholder not in text:
        return src
    out = hdinitpk.user_data_path('resolved_params', name)
    with open(out, 'w', encoding='utf-8') as f:
        f.write(text.replace(placeholder, hdinitpk.DATA_DIR))
    return out

The settings below are shared by every configuration. We marginalize over
$H_0$ rather than $\theta$ throughout.

The CMB-HD mock data (the fiducial spectra, the noise curves, the binning,
the lensing reconstruction noise, and the covariance matrices) all come
from [hdMockData](https://github.com/CMB-HD/hdMockData). We pin the version
rather than leaving it at hdfisher's default of `'latest'`, so that a new
release cannot quietly move the forecasts. v1.2 is what the paper used.

The $\tau$ prior of 0.005 is applied to every matrix. Where baryonic
feedback is varied we also add a prior on $\log_{10}(T_\mathrm{AGN})$ of
0.06% of its fiducial value of 7.8.

In [ ]:
use_H0 = True
hd_data_version = 'v1.2'
hd_datalib = hd_data.HDMockData(version=hd_data_version)

tau_prior = {'tau': 0.005}
feedback_prior = {'tau': 0.005, 'HMCode_logT_AGN': 0.0006 * 7.8}

# k bin edges [Mpc^-1] for the 11-bin binned-P(k) scheme
pk_bin_edges = [3.667637511098258835e-03, 8.505899211272888866e-03,
                1.972668268698882926e-02, 4.574966151931515734e-02,
                1.061015459285717805e-01, 2.460682259622835044e-01,
                5.706756795889451617e-01, 1.323497700691443235e+00,
                3.069424940269466440e+00, 7.118538595893407539e+00,
                1.650914836730800417e+01, 3.828763111167506139e+01]

In [ ]:
nine_params = ['mnu', 'tau', 'logA', 'H0', 'ombh2', 'omch2', 'ns', 'nnu', 'nrun']
eight_params = ['mnu', 'tau', 'logA', 'H0', 'ombh2', 'omch2', 'ns', 'nnu']
feedback_params = nine_params + ['HMCode_logT_AGN', 'n_ksz', 'A_ksz']
w0wa_params = ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun', 'w', 'wa']

binned_pk_params = [f'Pk{i+1}' for i in range(11)] + ['H0', 'tau', 'ombh2', 'omch2']
binned_pk_params_so = [f'Pk{i+1}' for i in range(7)] + ['H0', 'tau', 'ombh2', 'omch2']
binned_pk_params_feedback = (binned_pk_params
                             + ['HMCode_logT_AGN', 'n_ksz', 'A_ksz'])

ext_model_params = {
    'lcdm_nrun':          ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun'],
    'lcdm_nrun_neff':     ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun', 'nnu'],
    'lcdm_nrun_neff_mnu': ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun',
                           'nnu', 'mnu'],
}

The CLASS parameter files use CLASS's own parameter names, so parameter
lists and priors have to be translated on the way into a CLASS Fisher
library. The matrices come back out with CLASS names, and the plotting
notebook translates them back.

In [ ]:
camb_to_class = {
    'ombh2': 'omega_b',   'omch2': 'omega_cdm',  'theta': 'theta_s_100',
    'tau': 'tau_reio',    'logA': 'ln_A_s_1e10', 'As': 'A_s',
    'ns': 'n_s',          'H0': 'H0',            'nnu': 'Neff',
    'mnu': 'sum_m_ncdm',  'nrun': 'alpha_s',     'omk': 'Omega_k',
    'w': 'w0_fld',        'wa': 'wa_fld',
    'HMCode_logT_AGN': 'log10T_heat_hmcode',
}

def to_class(p):
    """CAMB parameter names -> CLASS. Takes a list, or a dict of priors."""
    if isinstance(p, dict):
        return {camb_to_class.get(k, k): v for k, v in p.items()}
    return [camb_to_class.get(k, k) for k in p]

## Fisher Configuration 

`Fisher` takes one required argument, the directory it reads and writes.
Constructing it creates `fiducial_params.yaml` and `step_sizes.yaml` in
that directory, recording what the run was configured with, along with
`theory` and `derivs` subdirectories.

Here is the nine-parameter CMB-HD CLASS forecast, which is the one behind
most of the paper.

In [ ]:
class_9param = fisher.Fisher(
    os.path.join(deriv_dir, 'class_9param'),
    overwrite=False, use_H0=use_H0, use_class=True,
    hd_data_version=hd_data_version,
    fisher_steps_file=steps('class_fiducial_step_sizes.yaml'),
    param_file=params('class_fiducial_params.yaml'))

One thing to be careful about. `Fisher.__init__` creates an empty `derivs`
directory when it does not find one, rather than complaining. If it is
pointed somewhere wrong it will not error. It silently recomputes the
fiducial theory instead, which at these accuracy settings runs for hours
with no output and looks like it isnt running. It is worth checking what is there
before building anything.

In [1]:
def deriv_count(fisher_dir):
    d = os.path.join(fisher_dir, 'derivs')
    if not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))])

print(f'{deriv_count(class_9param.fisher_dir)} derivative files')

NameError: name 'class_9param' is not defined

`calculate_fisher_derivs()` steps every varied parameter up and down and
writes the derivatives. Uncomment it to run it here. Depending on your enviornment, it may work in the notebook (though it will be quite slow).

The CMB-HD delensing needs the lensing reconstruction noise, which hdfisher
loads from hdMockData at the `hd_data_version` set above. It is worth
checking that it found the version you meant.

In [ ]:
# What the delensing will use. `lensing_noise_spectrum` also takes
# `pol_only_lensing=True` for the EE/EB-only reconstruction, which goes with
# the polarization-only covariance matrix used further down.
print(hd_datalib.lensing_noise_fname())
print(hd_datalib.block_covmat_fname('lensed'))
print(hd_datalib.block_covmat_fname('lensed', pol_only_lensing=True))

# class_9param.calculate_fisher_derivs()

With the derivatives on disk, `get_fisher_errors` returns the marginalized
$1\sigma$ errors directly. It takes the same arguments as `get_fisher`,
which are the spectra to use, the parameters to include, the priors, and
whether to add the DESI BAO Fisher. Since this is a CLASS library, the
parameter list and the prior go through `to_class`, and the errors come
back with CLASS names.

In [ ]:
errs = class_9param.get_fisher_errors(
    'lensed', params=to_class(nine_params), priors=to_class(tau_prior),
    use_H0=use_H0, with_desi=True)

pd.Series(errs, name='CMB-HD + DESI, 9 parameters (CLASS)').to_frame().T

## The nine configurations

Three things distinguish them. The Boltzmann code, the experiment, and
whether the primordial spectrum is a power law or binned. Configurations
that use a binned $\mathcal{P}(k)$ or the kSZ template need
`hdinitpk.hdinitPkfisher.Fisher`, which subclasses hdfisher's and adds
`binned_pk`, `bin_edges`, `ksz` and `pk_frac_step`. The rest use hdfisher's
directly.

The two CLASS configurations, `class` for CMB-HD and `so` for SO-like, are
the ones behind Table IV and Figures 2 and 4. They take the CLASS step-size
and fiducial-parameter files, so the parameters they vary carry CLASS names
(`omega_b`, `tau_reio`, `ln_A_s_1e10` and so on), and so do the Fisher
matrices built from them. The `to_class` helper above translates a
parameter list or a prior dictionary on the way in.

The other seven run CAMB. The `camb` configuration is the CMB-HD
nine-parameter forecast used for the comparison with CLASS in Appendix B
and for the accuracy study of Appendix A. The two `w0wa` configurations use
CAMB because we found CAMB and CLASS disagree by up to about 45% for that
model. The `camb_feedback_ksz` configuration is the delensed forecast of
Table V. The three binned $\mathcal{P}(k)$ configurations use CAMB because
the binned spectrum goes through hdPk, which is CAMB only.

Every job is built with the same `hd_data_version`, which `calculate_derivs`
supplies below. That is what selects the mock spectra, the noise curves, the
lensing reconstruction noise used for the delensing, and the covariance
matrix.

In [ ]:
jobs = {
    # CMB-HD, CLASS. Behind Table IV and Figures 2 and 4.
    'class': dict(
        dirname='class_9param', cls=hdfisher_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=True,
            fisher_steps_file=steps('class_fiducial_step_sizes.yaml'),
            param_file=params('class_fiducial_params.yaml'))),

    # SO-like, CLASS. Backs the 9-parameter SO forecast and the three
    # extended models of Table IV.
    'so': dict(
        dirname='so_9param', cls=hdfisher_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=True, exp='so',
            fisher_steps_file=steps('class_fiducial_step_sizes.yaml'),
            param_file=params('class_fiducial_params.yaml'))),

    # CMB-HD, CAMB. For the comparison with CLASS (Appendix B) and the
    # accuracy study (Appendix A).
    'camb': dict(
        dirname='camb_9param', cls=hdfisher_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=False,
            fisher_steps_file=steps('camb_fiducial_step_sizes.yaml'),
            param_file=params('camb_fiducial_params.yaml'))),

    # 12 parameters, the nine plus baryonic feedback and the kSZ template.
    # Table V, with delensed spectra.
    'camb_feedback_ksz': dict(
        dirname='camb_feedback_ksz', cls=hdinitpk_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=False,
            binned_pk=False, ksz=True,
            fisher_steps_file=steps('fiducial_step_sizes_feedback_ksz.yaml'),
            param_file=params('fiducial_params_feedback_ksz.yaml'))),

    # w0waCDM + alpha_s, with CAMB. Both need the w0wa step-size file,
    # since hdfisher's default steps do not vary w and wa. The parameter
    # file's lmax of 24000 takes precedence over the experiment's, so the
    # SO theory is computed to 24000 and then binned into SO's ell ranges,
    # as it was for the published forecasts.
    'w0wa_hd': dict(
        dirname='hd_w0wa', cls=hdfisher_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=False,
            fisher_steps_file=steps('camb_w0wa_fiducial_step_sizes.yaml'),
            param_file=params('camb_w0wa_fiducial_params.yaml'))),

    'w0wa_so': dict(
        dirname='so_w0wa', cls=hdfisher_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=False, exp='so',
            fisher_steps_file=steps('camb_w0wa_fiducial_step_sizes.yaml'),
            param_file=params('camb_w0wa_fiducial_params.yaml'))),

    # Binned P(k), with CAMB through hdPk.
    'binned_pk_hd': dict(
        dirname='hd_binned_pk', cls=hdinitpk_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=False,
            binned_pk=True, ksz=False, bin_edges=pk_bin_edges,
            fisher_steps_file=steps('binned_pk_steps_5_percent.yaml'),
            param_file=params('hd_binned_pk_fiducial_params.yaml'))),

    'binned_pk_so': dict(
        dirname='so_binned_pk', cls=hdinitpk_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=False, exp='so',
            binned_pk=True, ksz=False, bin_edges=pk_bin_edges,
            fisher_steps_file=steps('binned_pk_steps_5_percent.yaml'),
            param_file=params('hd_binned_pk_fiducial_params.yaml'))),

    'binned_pk_hd_feedback': dict(
        dirname='hd_binned_pk_feedback', cls=hdinitpk_fisher.Fisher,
        kwargs=dict(
            overwrite=True, use_H0=use_H0, use_class=False,
            binned_pk=True, ksz=True, bin_edges=pk_bin_edges,
            fisher_steps_file=steps('binned_pk_steps_5_percent_feedback.yaml'),
            param_file=params('hd_binned_pk_fiducial_params_feedback.yaml'))),
}

pd.DataFrame([
    {'configuration': name,
     'class': job['cls'].__module__.split('.')[0] + '.Fisher',
     'exp': job['kwargs'].get('exp', 'hd'),
     'code': 'CLASS' if job['kwargs'].get('use_class') else 'CAMB',
     'binned P(k)': job['kwargs'].get('binned_pk', False),
     'kSZ': job['kwargs'].get('ksz', False),
     'derivatives in': job['dirname']}
    for name, job in jobs.items()]).set_index('configuration')

### Running the derivatives

The loop below does all nine. It takes many hours in a single process, so
it is commented out. The same nine configurations are in
`run_hdInitPk_forecasts.py`, which runs them under MPI.

```
mpirun -np 4 python run_hdInitPk_forecasts.py
```

Set `CALCULATE_DERIVS = True` at the top of that file. Either way the output
lands in the same place, and the rest of this notebook picks it up.

Every configuration above is constructed with `overwrite=True`, so running
this discards any derivatives already in `deriv_dir`. That is what you want
for a clean rebuild and destructive otherwise.

In [ ]:
def calculate_derivs(name):
    job = jobs[name]
    out = os.path.join(deriv_dir, job['dirname'])
    print(f'{name} -> {out}', flush=True)
    kwargs = dict(job['kwargs'])
    kwargs.setdefault('hd_data_version', hd_data_version)
    fisherlib = job['cls'](out, **kwargs)
    fisherlib.calculate_fisher_derivs()
    return fisherlib

# for name in jobs:
#     calculate_derivs(name)

In [ ]:
# what is actually on disk
pd.DataFrame([
    {'configuration': name,
     'derivative files': deriv_count(os.path.join(deriv_dir, job['dirname']))}
    for name, job in jobs.items()]).set_index('configuration')

## The twenty Fisher matrices

Several configurations back more than one matrix. The `class` derivatives
give the nine-parameter CMB-HD forecast of Table IV, the three extended
models of the same table, and the eight-parameter matrix used to compare
against CAMB. The `camb` derivatives give the lensed and delensed
nine-parameter CAMB forecasts and the eight-parameter one used in the
accuracy study. The feedback binned $\mathcal{P}(k)$ derivatives are read
twice, once with the minimum-variance lensing reconstruction and once with
the polarization-only one (`pol_only_lensing=True`).

Four of the six SO matrices come from the CLASS `so` configuration, so
their parameter lists and priors go through `to_class` and the saved
matrices carry CLASS names. The other two stay on CAMB. `so_w0wa_lensed`
comes from the `w0wa_so` configuration, and the binned $\mathcal{P}(k)$
forecast cannot use CLASS.

All the forecasts use lensed spectra, apart from the two Table V matrices,
which use delensed spectra as in the paper. hdMockData carries CMB-HD data
only, so the SO covariance matrices come from hdfisher's own `data/covmats`
directory.

In [ ]:
matrices = [
    # (name, configuration, extra kwargs when reading, get_fisher_errors args)
    ('hd_camb_lensed_9param', 'camb', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=nine_params)),
    ('hd_camb_delensed_9param', 'camb', {},
     dict(cmb_type='delensed', priors=tau_prior, with_desi=True,
          params=nine_params)),
    ('hd_camb_lensed_8param', 'camb', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=eight_params)),

    ('hd_class_lensed_9param', 'class', {},
     dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
          params=to_class(nine_params))),
    ('hd_class_lensed_8param', 'class', {},
     dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
          params=to_class(eight_params))),

    ('so_class_lensed_9param', 'so', {},
     dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
          params=to_class(nine_params))),

    ('hd_camb_delensed_12param_feedback', 'camb_feedback_ksz', {},
     dict(cmb_type='delensed', priors=feedback_prior, with_desi=True,
          params=feedback_params)),
    ('hd_camb_delensed_9param_from_feedback', 'camb_feedback_ksz', {},
     dict(cmb_type='delensed', priors=tau_prior, with_desi=True,
          params=nine_params)),

    ('hd_w0wa_lensed', 'w0wa_hd', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=w0wa_params)),
    ('so_w0wa_lensed', 'w0wa_so', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=True,
          params=w0wa_params)),

    ('hd_binned_pk_lensed_11bins', 'binned_pk_hd', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=False,
          params=binned_pk_params)),
    ('so_binned_pk_lensed_7bins', 'binned_pk_so', {},
     dict(cmb_type='lensed', priors=tau_prior, with_desi=False,
          params=binned_pk_params_so)),

    ('hd_binned_pk_feedback_lensed', 'binned_pk_hd_feedback', {},
     dict(cmb_type='lensed', priors=feedback_prior, with_desi=True,
          params=binned_pk_params_feedback)),
    ('hd_binned_pk_feedback_pol_lensed', 'binned_pk_hd_feedback',
     dict(pol_only_lensing=True),
     dict(cmb_type='lensed', priors=feedback_prior, with_desi=True,
          params=binned_pk_params_feedback)),
]

# LCDM + alpha_s, then + N_eff, then + sum m_nu, for CMB-HD and SO
for model, model_params in ext_model_params.items():
    matrices.append(
        (f'hd_ext_lensed_{model}', 'class', {},
         dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
              params=to_class(model_params))))
    matrices.append(
        (f'so_ext_lensed_{model}', 'so',
         dict(fisher_params=to_class(['ombh2', 'omch2', 'logA', 'ns', 'tau',
                                      'H0', 'nnu', 'mnu', 'nrun'])),
         dict(cmb_type='lensed', priors=to_class(tau_prior), with_desi=True,
              params=to_class(model_params))))

print(f'{len(matrices)} Fisher matrices from {len(jobs)} configurations')

Each one is a `Fisher` built on that configuration's derivatives, then a
call to `get_fisher_errors`. Every library is opened with `overwrite=False`
here. The configurations above carry `overwrite=True` for the derivative
run, and reusing them as they are would clear the derivatives we are about
to read.

In [ ]:
def fisher_errors(name, config, extra, call):
    """Marginalized 1-sigma errors for one of the matrices above."""
    job = jobs[config]
    kwargs = dict(job['kwargs'], overwrite=False)
    kwargs.setdefault('hd_data_version', hd_data_version)
    kwargs.update(extra)
    fisherlib = job['cls'](os.path.join(deriv_dir, job['dirname']), **kwargs)
    return fisherlib.get_fisher_errors(use_H0=use_H0, **call)


forecast_errors = {}
for name, config, extra, call in matrices:
    try:
        forecast_errors[name] = fisher_errors(name, config, extra, call)
        print(f'  {name}')
    except Exception as err:
        print(f'  {name}: {type(err).__name__}: {err}')

print(f'\n{len(forecast_errors)} of {len(matrices)} built')

These are the forecasted $1\sigma$ errors that go into the figures and
tables. Rows are the matrices and columns the parameters. A blank means the
parameter is not in that model.

In [ ]:
pd.DataFrame(forecast_errors).T

## Comparing against the saved forecasts

The same twenty matrices ship in `hdinitpk/data/fisher_matrices`, which is
what the plotting notebook reads. The ratio of the errors just computed to
the saved ones should be 1 everywhere.

In [ ]:
def saved_errors(name):
    path = os.path.join(saved_matrix_dir, f'{name}.txt')
    if not os.path.isfile(path):
        return None
    return fisher.get_fisher_errors(*fisher.load_fisher_matrix(path))


ratios = {}
for name in forecast_errors:
    saved = saved_errors(name)
    if saved is None:
        continue
    new = forecast_errors[name]
    ratios[name] = {p: new[p] / saved[p] for p in new
                    if p in saved and saved[p] != 0}

comparison = pd.DataFrame(ratios).T
print(f'{len(ratios)} matrices compared against {saved_matrix_dir}')
comparison

A ratio that is not exactly 1 is not automatically a problem.

Agreement at $10^{-6}$ or better means the derivatives reproduce the saved
matrix exactly. Boltzmann codes are deterministic, so that is what an
unchanged calculation looks like. Disagreement below a percent is about what
recomputing the derivatives with a different CAMB or CLASS build gives.


In [ ]:
def marg_and_unmarg(path):
    F, par = fisher.load_fisher_matrix(path)
    F = np.atleast_2d(np.asarray(F, float))
    inv = np.linalg.inv(F)
    return ({p: float(np.sqrt(inv[i, i])) for i, p in enumerate(par)},
            {p: float(1 / np.sqrt(F[i, i])) for i, p in enumerate(par)},
            np.linalg.cond(F))


rows = []
for name in ('hd_w0wa_lensed', 'so_w0wa_lensed'):
    path = os.path.join(saved_matrix_dir, f'{name}.txt')
    if not os.path.isfile(path):
        continue
    marg, unmarg, cond = marg_and_unmarg(path)
    for p in w0wa_params:
        if p in marg:
            rows.append({'matrix': name, 'parameter': p,
                         'marginalized': marg[p], 'unmarginalized': unmarg[p],
                         'cond(F)': cond})

pd.DataFrame(rows).set_index(['matrix', 'parameter'])

## Saving

Once the ratios read 1, these can replace the saved set. The file names
already match what `hdInitPk_plots.ipynb` expects, so nothing else changes.

In [ ]:
os.makedirs(matrix_out_dir, exist_ok=True)

for name, config, extra, call in matrices:
    if name not in forecast_errors:
        continue
    job = jobs[config]
    kwargs = dict(job['kwargs'], overwrite=False)
    kwargs.setdefault('hd_data_version', hd_data_version)
    kwargs.update(extra)
    fisherlib = job['cls'](os.path.join(deriv_dir, job['dirname']), **kwargs)
    matrix, matrix_params = fisherlib.get_fisher(use_H0=use_H0, save=False,
                                                 **call)
    fisher.save_fisher_matrix(os.path.join(matrix_out_dir, f'{name}.txt'),
                              matrix, matrix_params)

print(f'{len(forecast_errors)} matrices written to {matrix_out_dir}')

## Forecasting something else

To add a model the paper does not cover you need a fiducial-parameter file
and a step-size file in `hdinitpk/data/fisher_fid_params` and
`hdinitpk/data/fisher_steps` (every parameter in the step file needs a
fiducial value), then a new entry in `jobs`, a derivative run, and a new
entry in `matrices`.

For a binned $\mathcal{P}(k)$ forecast use `hdinitpk.hdinitPkfisher.Fisher`
with `binned_pk=True` and `bin_edges`. The parameter file then needs
`nkbins`, the bin centers `k1` to `kN`, the amplitudes `Pk1` to `PkN` (or
`eneg2tauPk1` and so on), and an `effective_ns_for_nonlinear` entry.
`example_calc_binnedPk_forecasts.py` in the repository root is one complete
binned forecast, with the derivatives, the matrix and the errors annotated
line by line, if it is easier to read as a script.